# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdfQtX40h28F+ppTOLPWMM9CuzbNh8NHi6yfDoYOjOHkx0ZLlstMiSVw9oL+G/f/dRJVVJJWN6Jjk58cxpbKset+69dd9VftwIkngazvqL5cae2AjniyTNRZKNYvU2leXboggn5Yc8nFdPstsiD6Pq2W0q/UkYz0bxNE3mIl8u4INQT88XeZjEftQTJ2GW98RRGMC/B/FStV74+W0UjnXzz/BxFI/iV+JI5jLIxSC+D9Mknss43xMfk2QWSXGYRP5Y3GfiJAl8AOR46B2enxx8EPuwlj6O2JffYLasM9rYhhXn0Hm00RV+PBFxkjcbHe6N8LXRxakncgp4yJLoXnqT5CGOEn8iU28Spp2u2PqLyPJ0bxQLeI026D96f7SM/XkI4ERLEQFYucw0uEdpeC9FNZSAoWBpSbrsiaxY4LoRYdktvAmKPOvxiH39xVbupzOZZ1vj5VY4YdgKxGqPVpTd+qmciGkSwdiZeLgNYc5Z4ac+LFziyFP4JuNBoak4G3wZXIiHNMwBMQoYgDpPbIDTJMn7zYWGU8KhxrlCBb7GfiYRTQYZ/HGGfzv6MzwFNMmO5yFMntftVt1TmRdpXHb9WxLGHT1kD0AwaJHl/gwZjgmGnV+J3b74nIZzP12K4FYGd3utCOxYy9SNhGokkpgZTIGmvvYQHbCyiqO2J9h/u2UWA101djMH7Broy9Ol8Qlf0ySFnTcRIWElgt7IhG39y3E8nM0gAmHS7NXDUbvNnoq0ul+Y0XQ0nGsifCEuwriQDjAay9EvwF0awvbYF9eyHyUPEjcWrlXWV8pT37iHAWj9eNm5w056SByFvrgGfvFzf7SBnJP6D+qNnIaxBNrcdAW0xO6yL+NJ9hACk442+uFiGY9RVJTgqJHb1o8vxbgMrbtZVowbBKHmNmMTQ7estcZFMCILNJta+PXzoEKrZhv5LZALELj0B6TL2hR/puPCz7Jqm77ui0PcniLw4yRGgYnvJiFQS5aCMdTiqnxE3FKN29iFp0vazdsmLnsvai+2xFDt5NVdhyRv6X1mzcd9FLciAwXIQNUSDLw06RnUyebeP6BqI4OT8CMJ2MC9nam5ySVqNvzexT70fQvhDZi1WOdhUGbglqs/bUM6cHmvMVS9MTS6eZ6REQC70RpczAzZsjTGQ4MaxpxBxcxv+mIo/TS4Fe6VEheAlFHKGUwHn3U9yhuxWTHPJo84X1J3l5opcdemVlTf5zUKSrRlTdI6OpddiijyFnXppTr0eDQ389EjLeD7tHYtaC2R10LmdgXCxHBvBAa3RYwqwH7PXWEM+79nd1RIatkl+rWmzMdXc9c4ls74UjR4fvtyw+/UJG/LzccSme05BWNmCGnXdjKFePueMkf5vo3VNkLZz7m7zF6/yxZzMPVqIrUQSL/AgGhCzY3XsGic1gzA4rZnngWS2nwnEx3JqV9EuZiC4zb2LbOksuaUNzhZIZcdhoDRqykZ+FG3ZLyy8X+HqHA2qomQl69Rr1KJGO6OXx+dfz07OT84Glx4R8cXMGCbNz2KYeU1R/Lk/PDgxPt6fvHr8PPB4cAGJ5fzBU4to0yq9h8OhgM1zUudzuZUFjvrkZGhyyACfmAouhymGOZpEQAKAHOWW3lkmrIXB19rMNIMNqJKTwVGvhj8cnw2OFqzk/ZqjI6n51+OB0NXf2No7DxP7hFEs+/llzX65fdeJtN61/PLTyXVV/VO8lslG44OLg/WWyb7c9DldHB5QN3+bXh+1uinxqO1ydzHXv2/ZUlcUmywuJVzmcIOp/CR+HJqE4vZooVkNZ5RNPNCHl51pbWu2xuJp2MZlxcHw0/e8PLg4/HZxzUHyFM/u7UiItyoDbGOIUrM8rOV+LUHd2P5cnD6WU1dmw3ofXH8RePGprEK+4E+SMNxkeOG+nwlDqMiy8F0/iO4jUV8l8GGo5XaNFuBsiYjmbg6/HR19utwFcqNRyQJCAoDW2qE9TBt9D48uRpeujeLid2A1291wSEHF845jVGx90OS3snUnvD0gP78cnzSEHm13nMfp67oqhbcDniTOUzw7e4rFtGYp74Uu8GKBTlHqi+rznaa5Q4pVl6kPloRilW8o6uLg8vj8zNvODiE2V6/3envlLYFGKViHsYFxisWMMKdXE5TUD0ikzOMYItOAibJHITO2I/8OJAimQo2QPIkApmEX/1RgD2ntMgvV8OBON4+15zqnQxQK10enw7Ory4VDH/SILyCtzAVKKtJJiLpZ1JMSvAZ396nwcHF5YfBwWVtkLfvYJBX4tRP7wSjGmCbRqBYOCwobsHWzsfSz8nWffsuY8R9kek4gYlOkhltyz+CSTUuZuIsycMp2FE4eSYu4WkkYfcPPlx99E7OPw73xDhJ0INDNS05yN+fyRxMk8HZwYcTkBFlW1a7u2jH/gHtgR0i2/CvQNRTbECk38PA/KqdlC2BqnMwlmcV1UFsYYiZ0g7Znq3BdX6B0IFLC5LFUguwf78aXAGIH1ZN+PdCFrI/GRss29atuXGszgjrsQpL94QfPfjLTBQAd0QADodHYvjvJ2EuKdSPkXwZhcCIGFOz1kT8hG22oCctauqHEVgv2Z8F6eSHEEbFke1luiw1Yy324iz7zGhVH5KUcZzB7CKWgcwyjN0bIUBBzsEoRn5jw1ilWmoSXalqeGMaGU1LSAX1bDOnaljaLj1REcKS/k1lAd+ZssUh23pOW7MnLCuj/FhaDj3RsAV0owq4ugIqvzFhapW6o1jb/7Y7C4w59+8kECLrTHpMBC+5279MC6ks5xUulnavXqH4DIo0lXGwBEd97i9uE+AzEBBfWbycAIfm0Pb04D88TVINH3DLG35ycv7VG34eIFWbbXZQ5u0Aa0xCzrhdnZ0cnx5fDo4wTM2z5yhOt7KFBMmuHRA1q0J3Nd4u5f9UI7BvFdDwpExy9suldFyAA4JgOo+m85wjnSUgUgFs/Lsnihg3KmqdwEAXcnyGYIOSgO2chsFdJE3oofE9KENA/VpQ2gtVAuUynMukyMnpHhJ6vvp5cDtJZuKioIRhuTjgwou6wkCtYzSp6HT56WIw/HR+cuR9+ExI3dkRP8K/r9/2iV74+dcP25m7szXH7muaxEI0w3IxuLyAbY0aGMc8jsM8BEn4E3wEj1Sll96AJMwx1Jejy5ZnWpBuzeU8AVlz6Wd34jL1lSQEpGOkbAuT3SJCcTlPYFyQRZjbzqExZSFu1HsPRaiFdFAXd+jV4mMvSIoYrQjgUp73Y5SMlTaBr9GjTJMoM/iNn3h+kHMMGDfcKGbvbq2HrQBxUjvzEUv/kOQGk0eM/5DedCa2h6p9JnQHzmqkSZZtLSI/h09zWMgc9Fg4DqMwXxIz2elVfyrzZd+RQabJG+nf0cZ9OJGJbkuzYgShnxXjTrp5jXn67R//dW+08S9/+a+bzR7B2qN2XXeX0Sj7idqJsmEfjb1Fp2uFMPDJ9R6w3A1CCAsm9HTFX5ANBao0aqKxOV6iqPPY/ZrISOayQ/EF1OoebEyKnVTYRXPHUTfwgYapFwtc4qjiQ6gl62mCcUxQ0KzgqXLC6hDGoPTxeaR9Wlm6ucgd2cIPsHOa5UojQeMYjdQ5iJgY8/8pbAqcJcx5kGjZF5e3IViyqbyHJpkF298L2FrIxVkxJx3QXirQxAomXx2FGM2G3SaD/OIDJZ7TWw7l6dJj+CriEGwtT3HNVDvUj1j/0sd/3na6/Vv57Xrv55sn71GDjGYj7SEH0E8aAfgiXAKDZHnd6HMAaQBjgPiK6I/2uE30BInOBKcSDzJwq25cp9NHqjqg7BmQWXNdgUqK75jHePBJmN2tyl0aA9WCrdCQ2arjnEyRlGWZy7AQPliBpnUBsjjvAJGuiQt589yA4koxhrIHNkcRcY0Pb09BxCSx9dhEwdOeeJRPVoi6GdlvLnc1mzaX7mjfmlysUOFChzOUXRtB7Q8WUgF4gXGx4Fmp9sMDeeQvSQWAeVd+vVcWal1Dgxtlp3QdAmsY+HHWrBQSHXfQmEspaFIB7gzufPK8raKkIACZD2oTi6d0XRJ6IxLIWBXoVIVPIsyyAkQVDEYezew2ge0F/hnofSXeqDIKdI8h2GDEOfg2qOsdwtstvyoMwZxsu+kqMkcFFJOgnss1xgCA18jsOkSjI0GrJ3PJwqrExpHkfdaGt0d+hd4xWDJxZpByoj1nxIaOAJo0ZTZb6J5W9QaoZrQaoHEnHW38J7u6P45GwB/9H/8JtTS0OP54dn4xODwYDrq91q5l7BF7I1ut6n9jpH0QWtBrmUzvKUQhxnKKNrTaMHu64fG0dAbB/gaMIm1ufaCrdtCRplYIFGfPgkKSOh6j3K90NjKBi14oF71Jo0aoNSCg+7mZRY1Wl0q0/ZuTlKn8ljnUPPVcOQ9m5+ywwgwMxurhX8SOWzSWmrpaYpuC1i+lzjAKMyX68yQ9jTxHl0pdHDJdb8QF0WYi7v0onFREJOoY0gwU66Ma98koMyxphRR/WephDZJhJyfRXLMZZGufbAXtaLJ26vHj/3b64TS9CqXfQ0Nr9e2ExGaVrl/f0ChnVUYGV5dWDiNtdRJ8lTXB45AsoYzqDn9hFQkoEW3MyflzR+1ma7FNo2CTEFuOU+elZukeQgQtER5bYDeNIHgCVM+D2w5D4SqscpEKE4xY3Uxqg0iCNVBTcGQRXzjUU4Nl8eX0r8qVNdtrbP9EYZzGaOAS35UKVre12dsB/qFqWCzEo+r0pDXgeLuyYxweGZdP6+jtRRHz4pWCwSrjeTIpoLmqfMcigCImDgtVqNcVdX3GooPpQP1tRMlsJtPWMn/wFNyl/VinaH2xuri/VtLfK41IFWGizltohoEngTyQpP4MMxwq+jGKPfi+Gbe40IELIPy4mHngaY0jOSlzBihhUDsP+Hu0Eie6kAP3mJ/hF9ATUxI9rOkARQyiAJMQE/SjYORUBsB9OD9sXFzINa4DrV9ay82NivV4HIX86H06Hl6eX/wV40o7Owry0rR3289l5EXmvJKOtRLbvObFoDkB/ha9vVe5FeqKGKQMi08BGAHGsuKgzIquzDjMZGOOH1HVYYlxg/PtxgA8QqdhNfZalaq5dqZpbqhcY3e0gdxcG5QCKJS7WeltMWPxMZWG9MKjK9V8MJlrlfhaswYIaRNmijTuKM0F2cIZsxysyk0T8A0UDBYtlEtWh7JiCs1DHQcP2dyhH4E/k+EpBR9jNxmwLUa4ycAkzYHsjywtOrI/66OUMYVS18kpFiM/xylWY6DAQh3YwRV5GBD0cw8lSJb780X9wIzCh5Yy/fJNnDxQ/Vo6xY+d0cYPn/Z+ON37YWicyfFQ9KUehpwSmB9Tkx0bR1/DBYhirLddyBgFiRiN8ARFgAFXFl9gkoN+w9RYJCi5SfIUkKp21flQ6Mimva1sVvVQpnnoqiKwMAuw/E8cYhQ/ij+9ow/09foZE1qjMW4uv+UcPqxFZcmppTYZmjfg3CUFCGC/yBPAvjqDlEpYR4A4KGIZB8mEJMpVHMJbWPWtn9JJl0jO/GApFFJXLRl0Rp+n6lfwdZ3Pp1GR3XbslXfU1IOY/k3TBOTs+ZDerC6tBOiBz8A4BdMk7VTToOVLK6Najh4HLNjDLvLp1s+moCEAEbUINKoa+NOnzhLEXACWFcKR7WM9DuINq+b6E6mf18yNBirKodsbWjh5oYiK5L2MvGw5H4NUpg+uoH10j6qRHveLxUKmejaMZuMzlM3HZ7+cjzZcYfdR8Xr3zZ9GxVTuTDXqZGT2Jcnb1nny85sJ/DvZmTQ6gzkDm+TrwcUZJ/Lx3fHZR7vy24Tjvb+zCo7BxcX5Resi/vlt4O42vDo8HAyH7R133tXkNn298/o1VzvS/pTzkKS2QYaeyP2ZejfHLPZMqk8gRQIQlGhWlqYLhedccbVDDD9ofYLTYI0KPzpYoDzLlO8AWz4sU1jjYjqVMFUlDtDQIn0QkorhM4IEAz1WO90R7WqV+TqMX62GxiQMWJxRhsZqZlvNo7WiS/jKMT7kUB52PGsJjR7toUYbZWsgqyiPTlbPCUh8Rm8aj4Fy1NGfNR4pUuJj9bbZGy1l7I4qjBSXAfKToS0MW7PvEzE7DlcOU05mU3S/69ZnDZfWyItk0dmxQhWwLo+rYtCreYSPTzdsnMFbbZGZcpJpkJMNyF2ypxvx2BRA3SfxqEaHdwpBT1VZML5ekfuUgpqZJVoZk85VsTZiJXwQywdiW0ozGWhzKXy3Dq7gZsUb12I2aI7a9rqNR/dZEzPO0SgmtobrPh/+oO2VLJDGVleUiHROUeuy/VJ/dTEqMW0pfp/2Vy3dbLn2kSiWcTFWbi07bqFmS65P4ex2C8iYpJj7jY2SLzFHdzTHeHlH1SsRuSWeX5756QSeZlj9xnY0u2a2eVrJWq22SNKW8FjyVWNdmdZFgPVEa61hyG3XhL4dxFK9rA0lux3rwHgExk+ITtQk9GdxkoGRRxumo3GaoAkbamxW3kg7uEpkrwKWVZWG9sFP47WA1XEy2s02WtdGpjYW1sQkWW9rAUf25m8CTVkfa8MGhqsppTtU77OHCXuujSErzhV/cLqhPBJLS5XjARmRVYcQJjCIbca36nVl5WB6yNI711sM403bGsjUdS+kXkOi4a6k1LyI8pDUABZhAJ8kU8eq1ltBleZywt8wYZoxVwMPo40ztp5iq3QVBk7SCQC+lBjPq7riGnRBUCmUa8fG62rG1sfyehO+2bzRSll/dmnmckZtQJQjaAsIx6mpanhMnzdvbH0N3yvGhU6WvqgsX1QkHH6mabtmSjllVra3Fin7zLBNEZe3IQb9luuR07JmaB4dz7yVETg0qs7p5WFNyos4Lzap3WVSpFEUjvup/HshsfTU/prETNW1GC/SBHXHmtFSR5yUb0UZVcDpwJf6yHFccxvOFoUXxtOEQyu2yLDIcRxnCwmyQHBZiACPf/KA138gj559OT46PqADFtjK3m44vG1rjzZwWqp+AswDqaEfV3HWjsrfp/7cm4+x1Y71gMLVKffW8zy5wgxYkbhv4LZP6RYPvl4UecfeD9ejjfg+BJ24lc1D9i23toBy6XIL4N1HeHvMi30qBOwxFJ4qndQ9WDbtB9l9L05uJZ5PgzdFjNWpGzc1qx9F375dpVErJ8PXAg9iomxY6GecaqHToUXezxZRiKXuPTyF13ACqDdY//viTT08CqS5NqlxQyE4aH69c+NsWlIEW2J+g1vv3tSzhNxc06ka9/XNmhEsQ3zgYEZ8NQZ1G3j+PdgwaJW0RFrpWogMcXD2ZXB2KG7lfbB9+/r9W7aM8W4bZF91Yw4ZPbDhwNZRUqE9dLWap2DV0+l8IWeKH/RsSPuK3D2Ms4EA2DeGORp8Obs6OXGJTwSe142inYjOoSlcUO3Bs/itV/YEEZY2fSKZyNvQEU84KMB+TeZJkdE2v6BK0VQXX8cYBazyTVkIH311sRILW1SDEoh0p0IFR2EWJLhzxKkPcN+Hvr556IiTkwBVOM8wVZYn4rVZgE0VnYLPPqntxEjOaJhSMhHdOWCRKrsBiz0w9TfRR2gy68IiEwJHSAOd58Pzs8Ori4vB2aWqlMeaYe0bUkwH5ClYd14nk9G0p7jJCyerS6LwhR36WiBzjNKQz0Y7rGDCJ8iFZh97K/dV7BHkAstnZkgMRVZPLmUW+dUDLV2It84ApabNQHOV64HJq/fQfqrPVHmPJYD6KPlTS/1jY3TkI6N2knlSdB7tZW7qVYK50W2MweexPFCybVk0q7niWisZWGvCVdJeFiV5tsdKEkRfpXbrpjbO9LizR7P1xC6/eVoxaHuptdWFix6x+Lx2LxbZVTqkzTofKVo/LUlkrp1+7vZIvJT2kGW+mRELBwTNKEW1DSZqf3sVOWgIEtYWWYwdoI18f5IJ7ifG0g9UWpxzLmqPmnkoS1aTvFZ3b1llYvhq4shx6g/xZGFWpwpdBxGbJwC7vefmbD1v2DbzioOQjvmr6WuuBBVmKNw8W5bqvgys9RKVKh5FmZT0O0JQxDo+lSZR7R2eYuhMW+5eAX7CWkgs1OFjfhWfrbr2CrtRLScQBN7XLrm4zfNF+w0y+FK6E7r2UxaWfPdBs8ead6E4i251sp8wIvFiMjQgA+nlCVdqKOUCwOdFpo8qjjbCSSRt+FUMhcP5QQJ+B5qqW9NUSn10xNxhDmW4cqMpcjXi6KVmQBPd1huNsHcp88u25TeNtqbfsEL1NfoZjkS9W2XRNnoxeqkTvWumA4DnvPIkay1232idyhkm2AGpXqOxM9D/HWLMPJD3UjFWHYl+qQhbY9ZnRJgxd7v4QqXXKsGej7nnTpVV7/FAsfTG9XrU17RxLKZ+ctdK0nilYOSRabXfIR1JJE6K+aKDmw5g6eFBRVDk+6+/P0CP78loRcnnqTiFEi4yniwSyjWSZYOz7rWaPNqctbS7OwKJLxUrpzSkH4tPl5efxefz4SUakh8Hl0JBwsgbp8lDBoKKfWlKO/r3SYgn65NiMo3Q1h/jGQ11CLkhpVS4rm4dOsN2toXIWgbI/ljr/PSo0WMZsU0mhJV4SkraQ+NLr6ghQRlHV7DqrYMZ3RSDkY7T5B9hFPnbeLa+8zXEo36ZOLsUuzv9nT8L+OL92z+Lb+/fdjHFG8mvcvxrmG+/e/PP/TfvRefXT5enJz0RhXdSfASHNemKw1uwqOQ2nYvE/8TQn/ppqLpYwZgKqoMA+Yoh8mEeFdrc5h3g7PIfW+wnbh0fNRWC3eHJ/ohJCMReyKXydMq1OYOB5HKbZLRPuroso9pkze6KDKAMDtk23rpcLqSuP2uusUFhaGeH+voXaivB17x19jWMPT3dvvpbA4hYvjYafCQJAp97pDXAzd9/s0PyIpXZwoESvgQK40WouTCv/nrHVXyNr3EyWfK9Sgu8+W7SKYtVVmGN106bpjTXsg6O1SxLMSOefdztnDipVUkruFE1gO/v4bnX0Ya+8a4UR01IVHkqpZTAaRycfB6QHkIPEiXLI57ymMgnDPFU+xbrhXG5AHuzZNgpDVyLubo4+f61kCnKif8JhW7KFmifyq72n1+2ZmDiGKy48mDT77RsdcCbVruO7HyuJr7E0G9YPuc8a+uvZmwsfM0Vt5jh2oDzcI8qx7Zyao0IJL60L8tddFixsrBpn5exL+LTg8/HxBFc8f2B7PKGMgMgvIW/xN32f8bydlq/isSqfqBG47JCLJC7QiO5vAqV0BokSTrBS0uSMoJk6O9uv9+vpaoyHU+zrSFw8PxFuK3M1W3NBHwurKSFXXuTUuEoXa6u3FPtS3S5xKnyBawCMnyBaRbO+KBH1V1/qfDcq5GrxtQKdbpuoYY7VaKApx+WogKkBXWwazcf9fRPm3+wkGbczFKbuyGYRht8xRazOk/Llwnp8to/i2wJTu6ENkTtUnVyRcd6R1jFOBxNa3rIhPD90ilu7G4O9FW7OwMoKmdOGcFzvP0hyNYzfJ0SgI1cuv9hkSazFItDkDWqC5DAnkWsN3Z6tcsVFGgdPz41Glxbu/5Gc7HDwFqXxw2XtieaDL4a46p9je97VXjC3iru5mqbVAW5bi5zyv9KNvzpjfikl7LHN76JV4/mjJuURgAkfdvsPoGYsJ4tgnyzJ8Au7u71d6dPP3TF/xN2i+kiwxbQFd65mIxOPjwrDZI7g6c5K66NBkfQ9Dm36nMSRSqtgm5zjDXKZbaM8yZKyGAGpMF2pXFCjo/FFxVY/1py137dFbaVVQvD6UmeE5u3fqZsp7oVsK56eAduok8Whqa/Ht2gPdJQ/CR2n7arp5Tj9TjNhMyRTMW/JWOz/9+SMfQmxtk0+oV5JGHETWgNzza7bTZHKp+P/OmEVcULPcHTUqlMT5RrUJ/BjGKY1TkirhzG6kA8jAnqdk9MgX+RusjY7WKrGON9SopdFBgorBHhgVxDaP0m04TXiA35XaNBuW5sU35oNLPRgW3tbxodTFRhc+uzy0xZV6ZalHTL1fWEmnyrbsMDlm5oa2DSQLF5iZQnk3wd5mHG6lO3bgO9QGj5FAdQF8VWHMv+NqNYMSyueK9WXNLDG6lyzcfIje28WH0efJMBXYeok7xbCAVedYg5XU70g1URZ3QsBQ+klDfa4UHhEj09U5nx2Z4F2I+ga4JKMWc9vtWGL4IQeLFqdgvT9Od391ak3MwXu6DGK135gCXTZp8wcs13vZrcaRjCyKbOPnWOvjHJN/WC6Uy1VVTTN8J6qhYI2OnxyaA5c0I1vt55xrglJ1WtjO13U6uMdhxHryGgRZRz2WWN76+HwCbiUTHL041g/1MMkyINtDYrr1Kg+ZDuR3xb0mNt5lb/ryyGsNesUtGNW3rqCzL3MGZI4Xk9guzMn8ICw5hywOUKvcdqahtc7szVJ2sNjazyzNh2hH1Kh2vKJfSqKeskc1xj87J8oXGfzW9NnlXP6GebDnGjIwH0LanNw9TVPUfgEKJ0MMRB/VoBm9K9isBde+bXffGhCKOJ+OUXrP4xxBBI3zlIUmMOGIGuTeML3dS21VLWeMhmcwyOmsUIQTptdIPvvHs/wh5vzITAvYcwBI32VETDz3iWT/3X79+JbfFp8OXQvmEi86hsCmU+/OU6Iz0sFYocYA/j26ovmm9UnYTVYo7KrZroUFM1ea0cx8EI956fzjhJtrEV7N3zcszCKKrDoutZ1JXri7fqy+Dv3B7jTYDB+m9DOdzb9hmjcPyN8MMjp1Nz4AYI9zJdIvdZgtYx3/ct3igX+59c/LvvWbzB1/MJj2sV0OEBhUmoyyuX6i8VaJZ70S56tPYXXfHLW6h+f8V8AsIrx+pnXMs9w2z0NYsarcbWMD8yRhqW5JbfTIfgt5i68mI586n+J890Dced9P4h08TVRwcOFAHDhdzbbbQrZXX19U1NRr3pixMfnPZbLaRU6SGHfihGUYUl1G2VawfkXr/3d/E6df5JIsMKm5aeV2VIoK9FKscyR9GPKk0X5UihB0WB3PYaMNGxjYRGbA+XaZdsfqZ8ChC1pw48m4WYn48/D1YUaFplnONiit7B/q6Fai4MGNPpQzSaXIl+LGf0FoHyxGrfTxeZ83u+eHafCo3t43HuH5HhG7djwoA6O+yKpmOrffrTLD7WL6wm5gZGxQzAuf+9v7lkLJRc0g4Pr2qZcdjr3RsNT0sC6jf/4hAf6m2si/C8YmUWMVbBvfaMaKwhi3jFKozGyUMbP5kvmKGDTbcsTqQacEwWr8AS7bJaKNSRnDZfz3rrzQ5ree/NbrBbqIPaOc+2p6DcXsloz7YngpY96NPzfcoiITNc2d7rqYVk+FodACAB2/CHWmSrVhngA+nl7/V3phSoFP/FNzTrZ7RQ97VJJWS2QAPestuSfHnwYQesuJPAlQGkjndhFNVZ+WVOYTnFpKA0UAtWNIFW5gIbviBGqRFMfk5K7Q94W7j72tnKZ8IG9dvQjKf7+43r0FYtmi8jaFtZpW+nfLbzhSpTXAP4eGBogm5zbbl4omxNXNEPqZELVtXaKye95ZeBjZjHigPStXjIihPSdQ+uxHivHlSx/V8V5DOca02zejfwkjp4Abm6h7wZwWtJvPGtEM9TUGNOdB41VJSAEKcfXm4KiaGPv8NVhak2akbhuz7HE5eNaGIt42PHopWkN6R3Hb29Eqe1Gd/3+SY0IzqHcYskxV9sMC79JLK8KCzhNjjWDFYoFmwJS7ywqs7YI/VEo75fnWgUJclCV/MaAVFHDS/XD/ARAf4lUIGXz8I2onN3SPbO3P8mXtN7/sUYeNJtlsNNwMPBa4yy6i5BfPFPfVuHIhq+6PN1dEQGtE+ySMpF543rbr3y4sAa9fACfRVLNzJgzQsoKaDsoHPLwsoFopfjPnzRwg6NxteKSDcqFNrs9lw6urIRXJdaYmcrgo6TlKyxr/62lZn+fmsz81C/bRqMXvjxsvqpAfyZ7Mws6ON7gKnspz5w/96PCryH0O2P8O255vgr1/qCGgH9Mjh5p/+uLhLcERKDDd23R2Kky2BVaIOWgvHVD+L1jss80K9VdmLDNjqGcffEV7DN9C2jVTLYPEpXd5odKHhjZXpASHB2uiGrSJGogidJKZuQcjHJAgxJjD8gJNaZPhRhqgMTvi602oMP5HKBKn7/c1MRP5Mdlj/vqEOKOL86bYbH3jiOQRayXWhTj8ivmAWeK5l9fKTHMbPkLxgHoNtbrfXrD1VtFT47/SC+XBycdl80I6fSPumjlAc6y4aKTzxu/lVmmypg2zyNS9cNbJ4looMHu39RP4Xa3XzZmo3f+9Frd56/fFLsC8oQ74LwY5kUWbTU6u9Fc/JvjZ0ktd8bwxXz9ZRHoqN+oaxL61cDGpc+8uKPjoeq+am++iWjW1XWx4LF1jVLbB1JZhpcxM32hjM3W4mrKphEVobrOgqyM2B3g7E642NIbrI0LDPreCNf5drhkxz7nDyvW0U9gWHV/Q5+0etiFTUYZrHr2EbOkZS6oaAWoW+4yC2kPGvsUC5Jn1IWdDfzFBxqR4Xc1cXJd9pJtRba9nGdnXQrwOenWM1kWhgG8O90d6dcry4tOKyWqTehUUPZGjFgtLqqZZ3xMKcWNQB31BKCK/Vm+sbSag26qINt8L/jKBvayuWpyUbAuLGYFTvugWHAC+fUKluHMlRp3ZbQrzbb+JX43ChQ4Os6fbxtNLccaXy9zFIDCfa7GmjuMGJLsExpEgWAuq2GBVk9WmZgcHenvEQGdrCnzjDPgQvUVcPq2kC+ktk43V2/JxnvAuSKN1ynsgLYLKguLTDMEe3k125hNqZTQCsZu2/draC3gO2qqksn0KSyLzr9VS7HCajhY/z1r7RYmOH7VcqjsnEmu3pNyhwJ9VB81XWRwRhieFvktJXw98Pq20FBt2ITqB/3VJoH78cj28TzuFbII7p4XpmDq1MML+Z5+v8Qv8J7"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
